# 🧠 Bidirectional and Stacked RNNs
**Course:** Deep Learning  
**Lecture:** 5 — Recurrent Neural Networks (RNNs)  
**Part:** c — Bidirectional & Stacked

---

> **Where we are:** We know how to process sequences left-to-right. Now we look at building deeper networks (stacking) and reading sequences backwards as well as forwards (bidirectional) for better context.

**What you'll learn:**
- Unidirectional vs Bidirectional processing
- How `Bidirectional` wrapping affects output shapes
- Stacking multiple RNN layers
- The combination: Stacked Bidirectional RNNs

**Exam relevance:** ⭐⭐ (Shape calculation and return_sequences usage are common questions)


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Confirm device
print(f"TensorFlow {tf.__version__} | Devices: {[d.name for d in tf.config.list_physical_devices()]}")

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12


## 📖 1. Architectures

1. **Unidirectional**: Reads sequence from $t=1$ to $T$. Good for real-time forecasting where the future is unknown.
2. **Bidirectional**: Runs two RNNs—one forward, one backward. Their outputs are usually concatenated. Excellent for text classification, translation (where full sequence is available).
3. **Stacked**: Layer $L$ uses the sequence of hidden states from Layer $L-1$ as input. Increases model depth and capacity.


## 🔀 Your Options: Network Topologies

At this step, you can choose between:

| Option | Pros | Cons | When to use |
|---|---|---|---|
| Unidirectional | Fast, causally correct | Misses future context | Time series forecasting |
| Bidirectional | Maximum context | Cannot be used real-time | Text analysis, NLP |
| Stacked | Deep hierarchical features | Harder to train, overfits | Complex large datasets |


In [ ]:
input_shape = (20, 8)  # (seq_length, features)

# 1. Unidirectional LSTM
model_uni = tf.keras.Sequential([
    tf.keras.layers.Input(shape=input_shape),
    tf.keras.layers.LSTM(32),              # return_sequences=False (default) - only need last state
    tf.keras.layers.Dense(1, activation='sigmoid')
], name='Unidirectional')

# 2. Bidirectional LSTM - reads forward AND backward, output doubles to 64
model_bi = tf.keras.Sequential([
    tf.keras.layers.Input(shape=input_shape),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),  # output: 32*2=64 units
    tf.keras.layers.Dense(1, activation='sigmoid')
], name='Bidirectional')

# 3. Stacked LSTM - first layer needs return_sequences=True to feed next layer
model_stacked = tf.keras.Sequential([
    tf.keras.layers.Input(shape=input_shape),
    tf.keras.layers.LSTM(32, return_sequences=True),  # returns (batch, 20, 32)
    tf.keras.layers.LSTM(32),                          # consumes (batch, 20, 32), outputs (batch, 32)
    tf.keras.layers.Dense(1, activation='sigmoid')
], name='Stacked')

# 4. Stacked Bidirectional
model_stacked_bi = tf.keras.Sequential([
    tf.keras.layers.Input(shape=input_shape),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32, return_sequences=True)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dense(1, activation='sigmoid')
], name='Stacked_Bidirectional')

# Print parameter counts
for model in [model_uni, model_bi, model_stacked, model_stacked_bi]:
    total = model.count_params()
    print(f'{model.name:25s}: {total:,} parameters')

In [ ]:
# ✅ Solution: Build the Architectures

input_shape = (20, 8)

model_uni = tf.keras.Sequential([
    tf.keras.layers.LSTM(32, input_shape=input_shape)
])

# Bidirectional wraps the LSTM
model_bi = tf.keras.Sequential([
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32), input_shape=input_shape)
])

# Stacked requires return_sequences=True on the first
model_stacked = tf.keras.Sequential([
    tf.keras.layers.LSTM(32, return_sequences=True, input_shape=input_shape),
    tf.keras.layers.LSTM(32)
])

model_stacked_bi = tf.keras.Sequential([
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32, return_sequences=True), input_shape=input_shape),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32))
])

print("Unidirectional Output Shape:", model_uni.output_shape)
print("Bidirectional Output Shape: ", model_bi.output_shape)
print("Stacked Output Shape:       ", model_stacked.output_shape)
print("Stacked Bi Output Shape:    ", model_stacked_bi.output_shape)



In [ ]:
# 🔬 Comparison: Training the Variants

# Load IMDB dataset for sentiment analysis (built-in Keras)
vocab_size = 5000
max_len = 100

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=vocab_size)
X_train = tf.keras.preprocessing.sequence.pad_sequences(X_train, maxlen=max_len)
X_test = tf.keras.preprocessing.sequence.pad_sequences(X_test, maxlen=max_len)

# Reduce dataset size for quick training in this notebook
X_train, y_train = X_train[:2000], y_train[:2000]
X_test, y_test = X_test[:500], y_test[:500]

def build_full_model(base_model_layers):
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(vocab_size, 32, input_length=max_len)
    ])
    for layer in base_model_layers:
        model.add(layer)
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

variants = {
    'Unidirectional': [tf.keras.layers.LSTM(16)],
    'Bidirectional': [tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(16))],
    'Stacked': [tf.keras.layers.LSTM(16, return_sequences=True), tf.keras.layers.LSTM(16)],
    'Stacked_Bi': [tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(16, return_sequences=True)),
                   tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(16))]
}

histories = {}

for name, layers in variants.items():
    print(f"Training {name}...")
    model = build_full_model(layers)
    hist = model.fit(X_train, y_train, epochs=3, validation_data=(X_test, y_test), batch_size=64, verbose=0)
    histories[name] = {
        'val_acc': hist.history['val_accuracy'],
        'params': model.count_params()
    }

plt.figure(figsize=(10, 5))
for name, data in histories.items():
    plt.plot(data['val_acc'], label=f"{name} (Params: {data['params']})")
plt.title('Validation Accuracy on IMDB Subset')
plt.xlabel('Epochs')
plt.ylabel('Val Accuracy')
plt.legend()
plt.show()

print("\n| Architecture | # Params | Final Val Accuracy | Description |")
print("|---|---|---|---|")
for name, data in histories.items():
    print(f"| {name} | {data['params']} | {data['val_acc'][-1]:.3f} | ... |")



## 🎓 Exam Corner

### Question 1 (Conceptual)
If `LSTM(64)` produces output of shape `(batch, 64)`, what does `Bidirectional(LSTM(64))` produce?

<details><summary>💡 Answer</summary>

It produces `(batch, 128)`. The Bidirectional wrapper concatenates the output of the forward LSTM (64 units) and the backward LSTM (64 units) by default.
</details>

### Question 2 (Practical)
What type of task is a Bidirectional RNN NOT suitable for, and why?

<details><summary>💡 Answer</summary>

It is **NOT** suitable for autoregressive tasks (like language modeling / next-word prediction) or real-time time-series forecasting. Why? Because a bidirectional RNN looks into the future to process the current timestep. If you are trying to predict the future, you cannot allow the model to have already seen it (data leakage).
</details>
